# 02 — Chunking & Embedding

Sanity-check the RAG pipeline before scaling up.

**Chunking decision**: KB rows are already legal units (Chương → Điều → Khoản → điểm), avg ~284 chars, max 1752. Short enough to embed as-is, so `chunk_passage` is a passthrough.

**Embedder**: `intfloat/multilingual-e5-base` with `query:` / `passage:` prefixes, L2-normalized vectors, FAISS `IndexFlatIP` (= cosine).

**Steps**:
1. Load the KB (600 passages).
2. Eyeball passage-length distribution.
3. Build / load FAISS index.
4. Run a few Vietnamese queries and inspect top-5.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.config import EMBEDDING_MODEL, INDEX_DIR, KB_CSV
from src.data.loader import load_knowledge_base

passages = load_knowledge_base(KB_CSV)
print(f"{len(passages)} passages")
print(passages[0])

In [ ]:
import statistics
from collections import Counter

lens = [len(p["passage_text"]) for p in passages]
print(f"min={min(lens)}  avg={statistics.mean(lens):.1f}  max={max(lens)}  median={statistics.median(lens)}")

doc_counts = Counter(p["doc_id"] for p in passages)
for doc_id, n in sorted(doc_counts.items()):
    print(f"  {doc_id}: {n} passages")

In [ ]:
# Build (or rebuild) the FAISS index. On Kaggle / GPU this is ~10s.
# Skip this cell if you already ran `python scripts/build_index.py`.
from src.rag.embeddings import get_embedder
from src.rag.vectorstore import build_index, load_index

if not (INDEX_DIR / "kb.faiss").exists():
    embedder = get_embedder(EMBEDDING_MODEL)
    build_index(passages, embedder, INDEX_DIR)

index, meta = load_index(INDEX_DIR)
print(f"index size: {index.ntotal}, meta rows: {len(meta)}")

In [ ]:
# Smoke-test retrieval with a handful of representative Vietnamese queries
from src.rag.retriever import retrieve

embedder = get_embedder(EMBEDDING_MODEL)

queries = [
    "Thuế suất thuế giá trị gia tăng đối với hàng hóa xuất khẩu là bao nhiêu?",
    "Thu nhập chịu thuế thu nhập cá nhân gồm những khoản nào?",
    "Đối tượng nào được miễn thuế sử dụng đất phi nông nghiệp?",
    "Hàng hóa nào chịu thuế tiêu thụ đặc biệt?",
    "Thuế suất thuế thu nhập doanh nghiệp phổ thông hiện nay là bao nhiêu?",
]

for q in queries:
    print("Q:", q)
    for hit in retrieve(q, index, meta, embedder, top_k=3):
        snippet = hit.passage_text[:120].replace("\n", " ")
        print(f"  #{hit.rank} score={hit.score:.3f} [{hit.passage_id}] {snippet}...")
    print()